# Swarm-Optimized Neural Network Ensembles
**Multi-Agent Systems — Final Project**

This notebook is a thin driver. All logic lives in the `.py` modules on Drive.
The only cell you should need to edit between runs is **Section 3 — Configuration**.

---
**Sections**
1. Environment Setup
2. GPU Check
3. Configuration ← edit this between runs
4. Smoke Test
5. Full Ablation Run
6. Results & Plots
7. Hyperparameter Sweep (optional)

## 1 — Environment Setup

In [ ]:
# Mount Google Drive — checkpoints and data live here
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo from GitHub — source code lives here, not on Drive
# First run: clones. Subsequent runs: pulls latest changes.
import os
if not os.path.exists('/content/multi-agent-ml'):
    !git clone -b implementation https://github.com/blakedehaas/multi-agent-ml.git
else:
    !git -C /content/multi-agent-ml pull origin implementation

%cd /content/multi-agent-ml
branch_name = !git rev-parse --abbrev-ref HEAD
print('Repo ready. Branch:', branch_name[0])

In [ ]:
# ── Pull latest changes from GitHub ───────────────────────────────────────
# Run this cell any time there is an update mid-session.
# Autoreload handles the rest — no restart needed.
!git pull origin implementation

In [ ]:
import sys
from pathlib import Path

# ── Per-collaborator setting — update this to match your own Drive path ───
# Oscar:
DRIVE_ROOT = Path('/content/drive/MyDrive/Final_Project')
# Blake — uncomment and set your path:
# DRIVE_ROOT = Path('/content/drive/MyDrive/...')

# ── Source code: cloned from GitHub (same for everyone) ───────────────────
PROJECT_ROOT = Path('/content/multi-agent-ml')

# ── Data and outputs ───────────────────────────────────────────────────────
DATA_ROOT      = Path('/content/data')                        # CIFAR-10 — re-downloads each session
CHECKPOINT_DIR = DRIVE_ROOT / 'experiments' / 'checkpoints'

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Source       : {PROJECT_ROOT}  (GitHub)')
print(f'Checkpoints  : {CHECKPOINT_DIR}  (Drive)')
print(f'Data         : {DATA_ROOT}')
print(f'Source exists: {PROJECT_ROOT.exists()}')

In [ ]:
# Install dependencies from requirements.txt
# Upgrade IPython first — required for autoreload compatibility on Python 3.12
# torch and torchvision are pre-installed on Colab — pip skips them automatically
%pip install -q --upgrade ipython
%pip install -q -r requirements.txt

In [ ]:
# Enable autoreload — picks up changes to .py modules after a git pull
# automatically, no session restart needed.
%load_ext autoreload
%autoreload 2

In [ ]:
import shutil

for pycache in PROJECT_ROOT.rglob('__pycache__'):
    shutil.rmtree(pycache)
    print(f'Removed {pycache}')

print('pycache cleared.')

In [5]:
# Log in to W&B — paste your API key when prompted
# Get your key at: https://wandb.ai/settings
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: osro6012 (osro6012-university-of-colorado-boulder) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 2 — GPU Check

In [4]:
import torch

assert torch.cuda.is_available(), (
    'No GPU detected! Go to Runtime → Change runtime type → GPU (A100)'
)

gpu_name   = torch.cuda.get_device_name(0)
gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f'GPU          : {gpu_name}')
print(f'Memory       : {gpu_memory:.1f} GB')
print(f'PyTorch      : {torch.__version__}')
print(f'CUDA         : {torch.version.cuda}')

DEVICE = 'cuda'

GPU          : Tesla T4
Memory       : 15.6 GB
PyTorch      : 2.10.0+cu128
CUDA         : 12.8


## 3 — Configuration

**This is the only cell you need to edit between runs.**

Key knobs:
- `conditions` — which ablation cells to run (`None` = all 8)
- `epochs` — 50 is a good default; increase to 100 for a final run
- `alpha`, `beta`, `gamma` — rule strengths when active
- `k` — neighborhood size (3 = sparse, 9 = full connectivity for N=10)

In [5]:
from experiments.ablation import AblationConfig

cfg = AblationConfig(
    # ── Swarm rule strengths ──────────────────────────────────────────
    alpha = 0.3,     # gradient alignment
    beta  = 0.5,     # separation  → equilibrium spacing = β/γ = 5.0
    gamma = 0.1,     # cohesion
    k     = 3,       # k-NN neighborhood size

    # ── Training ─────────────────────────────────────────────────────
    n_agents    = 10,
    epochs      = 50,
    batch_size  = 512,
    subset_size = None,     # None = full CIFAR-10 (50k)
    device      = DEVICE,
    lr          = 1e-3,
    weight_decay= 1e-4,

    # ── Logging ───────────────────────────────────────────────────────
    cka_interval     = 5,       # compute CKA every N epochs
    landscape_at_end = True,    # loss landscape after training
    wandb_project    = 'swarm-optimization',
    wandb_mode       = 'online',

    # ── Conditions ───────────────────────────────────────────────────
    # None = run all 8. Or specify a subset:
    # conditions = ['baseline', 'separation', 'full_swarm']
    conditions = None,

    # ── Paths ─────────────────────────────────────────────────────────
    checkpoint_dir = CHECKPOINT_DIR,
)

print(f'Conditions   : {cfg.conditions or "all 8"}')
print(f'Agents       : {cfg.n_agents}')
print(f'Epochs       : {cfg.epochs}')
print(f'Dataset      : {"full CIFAR-10" if cfg.subset_size is None else f"{cfg.subset_size} samples"}')
print(f'Rules        : α={cfg.alpha}  β={cfg.beta}  γ={cfg.gamma}  k={cfg.k}')

Conditions   : all 8
Agents       : 10
Epochs       : 50
Dataset      : full CIFAR-10
Rules        : α=0.3  β=0.5  γ=0.1  k=3


## 4 — Smoke Test

Run 3 agents × 2 epochs × 2 conditions before committing to the full run.
This confirms the environment, Drive paths, and imports all work correctly.

**Expected output:** two conditions complete, summary table prints, no errors.

In [6]:
from experiments.ablation import run_ablation, AblationConfig

smoke_cfg = AblationConfig(
    landscape_grid_size=11,
    landscape_alpha_range=(-0.5, 0.5),
    n_agents       = 3,
    epochs         = 50,
    subset_size    = 1000,
    device         = DEVICE,
    cka_interval   = 1,
    landscape_at_end = True,
    wandb_mode     = 'disabled',
    checkpoint_dir = CHECKPOINT_DIR / 'smoke',
    conditions     = ['baseline', 'full_swarm'],
    # inherit rule strengths from main cfg
    alpha = cfg.alpha,
    beta  = cfg.beta,
    gamma = cfg.gamma,
)

smoke_results = run_ablation(smoke_cfg)
print('\n✓ Smoke test passed — safe to proceed with full run.')

Running 2 ablation condition(s): ['baseline', 'full_swarm']

────────────────────────────────────────────────────────────
Condition: baseline  (α=0, β=0, γ=0)
────────────────────────────────────────────────────────────

Run: baseline
Trainer: <baselines.single_trainer.IndependentEnsembleTrainer object at 0x7c5e21678110>
Agents: 3 × TinyNet (94,762 params each)
Train batches: 2  Val batches: 1



Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

  Epoch   1:   0%|          | 0/2 [00:00<?, ?batch/s]

Epoch   0  train=2.4330  val=2.3571  ens=2.2900  acc=0.180


  Epoch   2:   0%|          | 0/2 [00:00<?, ?batch/s]

...

Epoch  47  train=1.3624  val=1.7145  ens=1.6847  acc=0.360


  Epoch  49:   0%|          | 0/2 [00:00<?, ?batch/s]

Epoch  48  train=1.3455  val=1.7246  ens=1.6962  acc=0.350


  Epoch  50:   0%|          | 0/2 [00:00<?, ?batch/s]

Epoch  49  train=1.3373  val=1.7200  ens=1.6909  acc=0.350

Test  ensemble=1.6162  acc=0.416  best=1.6037  diversity=9.6778
Checkpoint saved → /content/drive/MyDrive/Final_Project/experiments/checkpoints/smoke/full_swarm.pt

────────────────────────────────────────────────────────────
Computing loss landscapes (shared color scale)...
  Grid: baseline
  Grid: full_swarm
  Landscape saved → /content/drive/MyDrive/Final_Project/experiments/checkpoints/smoke/baseline_landscape.png
  PCA saved → /content/drive/MyDrive/Final_Project/experiments/checkpoints/smoke/baseline_pca.png
  Landscape saved → /content/drive/MyDrive/Final_Project/experiments/checkpoints/smoke/full_swarm_landscape.png
  PCA saved → /content/drive/MyDrive/Final_Project/experiments/checkpoints/smoke/full_swarm_pca.png

Ablation complete.

Condition        Ens Loss    Ens Acc   Best Acc  Diversity    GAP CKA
────────────────────────────────────────────────────────────────────
baseline           1.5879      0.430      0.424 

## 5 — Full Ablation Run

Trains all configured conditions sequentially.
Metrics stream live to your W&B dashboard at:
**https://wandb.ai/osro6012-university-of-colorado-boulder/swarm-optimization**

Estimated time on A100 (50 epochs, 10 agents, full CIFAR-10): ~2-3 hours total.

In [8]:
# Pass data_root so CIFAR-10 downloads to local Colab storage
# We patch the default at runtime to avoid editing the source files
import data.cifar as cifar_module
cifar_module._DEFAULT_DATA_ROOT = DATA_ROOT

results = run_ablation(cfg)

Running 8 ablation condition(s): ['baseline', 'alignment', 'separation', 'cohesion', 'align_sep', 'align_coh', 'sep_coh', 'full_swarm']

────────────────────────────────────────────────────────────
Condition: baseline  (α=0, β=0, γ=0)
────────────────────────────────────────────────────────────



Run: baseline
Trainer: <baselines.single_trainer.IndependentEnsembleTrainer object at 0x7cf928d38950>
Agents: 10 × TinyNet (94,762 params each)
Train batches: 88  Val batches: 10



Training:   0%|          | 0/50 [00:00<?, ?epoch/s]

  Epoch   1:   0%|          | 0/88 [00:00<?, ?batch/s]

KeyboardInterrupt: 

## 6 — Results & Plots

In [ ]:
# Summary comparison table
from experiments.ablation import compare_conditions
compare_conditions(results)

In [ ]:
import matplotlib.pyplot as plt
from visualization.plots import plot_diversity_curves, plot_training_curves

# ── Diversity curves: one plot per condition ──────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (label, res) in zip(axes, results.items()):
    if res['cka_history']:
        fig_cka = plot_diversity_curves(res['cka_history'])
        # Redraw onto subplot
        src_ax = fig_cka.axes[0]
        for line in src_ax.lines:
            ax.plot(line.get_xdata(), line.get_ydata(),
                    label=line.get_label(), linewidth=line.get_linewidth())
        ax.set_title(label, fontsize=10)
        ax.set_ylim(0, 1.05)
        ax.set_xlabel('Epoch', fontsize=8)
        ax.set_ylabel('Mean CKA', fontsize=8)
        ax.grid(True, alpha=0.3)
        plt.close(fig_cka)

fig.suptitle('Representational Diversity by Condition', fontsize=13)
fig.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / 'diversity_curves.png'), dpi=150)
plt.show()

In [ ]:
# ── Training loss curves side by side ─────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

for ax, (label, res) in zip(axes, results.items()):
    epochs_  = [d['epoch']     for d in res['history']]
    means    = [d['mean_loss'] for d in res['history']]
    ax.plot(epochs_, means, linewidth=2, color='black', label='mean')
    for i in range(len(res['history'][0]['agent_losses'])):
        agent_curve = [d['agent_losses'][i] for d in res['history']]
        ax.plot(epochs_, agent_curve, linewidth=0.8, alpha=0.4)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel('Epoch', fontsize=8)
    ax.set_ylabel('Loss', fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Training Loss by Condition', fontsize=13)
fig.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / 'training_curves.png'), dpi=150)
plt.show()

In [ ]:
# ── Final CKA matrix for each condition at gap layer ──────────────────────
import matplotlib.colors as mcolors
from visualization.plots import plot_cka_matrix

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for ax, (label, res) in zip(axes, results.items()):
    if res['cka_history']:
        last   = res['cka_history'][-1]
        matrix = last['gap']['matrix']
        fig_cka = plot_cka_matrix(matrix, layer='gap', epoch=last['epoch'])
        src_ax  = fig_cka.axes[0]
        img     = src_ax.images[0]
        ax.imshow(img.get_array(), vmin=0, vmax=1, cmap='viridis', aspect='auto')
        ax.set_title(label, fontsize=10)
        ax.axis('off')
        plt.close(fig_cka)

# Shared colorbar — same vmin/vmax for all subplots so values are comparable
sm = plt.cm.ScalarMappable(
    cmap='viridis',
    norm=mcolors.Normalize(vmin=0, vmax=1),
)
sm.set_array([])
fig.colorbar(sm, ax=axes.tolist(), label='CKA Similarity', shrink=0.6, pad=0.02)

fig.suptitle('Final CKA Similarity (GAP layer)', fontsize=13)
fig.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / 'cka_matrices.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7 — Hyperparameter Sweep (Optional)

Run this section only after the ablation has identified a winning condition.
The sweep fine-tunes the rule strengths for that condition using Bayesian optimization.

**Step 1** creates the sweep on W&B (run once).
**Step 2** launches the agent (can duplicate this cell for parallel agents).

In [ ]:
# ── Step 1: Create sweep — run ONCE, then comment out ────────────────────
from experiments.hyperparam_sweep import create_sweep

SWEEP_CONDITION = 'full_swarm'   # ← change to your winning condition

sweep_id = create_sweep(
    condition = SWEEP_CONDITION,
    project   = 'swarm-optimization',
    method    = 'bayes',
    n_trials  = 30,
)
print(f'sweep_id = {sweep_id!r}  ← paste into Step 2')

In [ ]:
# ── Step 2: Launch agent ──────────────────────────────────────────────────
from experiments.hyperparam_sweep import run_sweep_agent

SWEEP_ID = ''   # ← paste sweep_id from Step 1

run_sweep_agent(
    sweep_id       = SWEEP_ID,
    condition      = SWEEP_CONDITION,
    project        = 'swarm-optimization',
    n_agents       = cfg.n_agents,
    epochs         = cfg.epochs,
    subset_size    = cfg.subset_size,
    device         = DEVICE,
    checkpoint_dir = CHECKPOINT_DIR / 'sweep',
    count          = 10,   # runs per agent process
)

## 8 — Topology Sweep

Tests whether the ratio of neighborhood size (k) to population size (n_agents) affects training stability.
Runs only `sep_coh` and `full_swarm` — the two conditions that showed instability in the full ablation.

| Config | n | k | k divides n? | k/n ratio |
|--------|---|---|--------------|-----------|
| n9_k3  | 9 | 3 | ✅ | 0.33 |
| n10_k3 | 10 | 3 | ❌ | 0.30 ← baseline |
| n12_k3 | 12 | 3 | ✅ | 0.25 |
| n12_k4 | 12 | 4 | ✅ | 0.33 |

If instability disappears when k divides n, graph asymmetry is the cause.
If instability persists across all configs, the β/γ force imbalance is the cause.

In [ ]:
# ── Topology sweep: does k dividing n_agents affect stability? ───────────
# Runs only sep_coh + full_swarm (the unstable conditions) across 4 (n, k) configs.
# α, β, γ are unchanged from the main ablation.

from experiments.ablation import run_ablation, AblationConfig

topology_configs = [
    dict(n_agents=9,  k=3, label='n9_k3'),   # k divides n  ✅  ratio=0.33
    dict(n_agents=10, k=3, label='n10_k3'),  # k divides n  ❌  ratio=0.30  ← current
    dict(n_agents=12, k=3, label='n12_k3'),  # k divides n  ✅  ratio=0.25
    dict(n_agents=12, k=4, label='n12_k4'),  # k divides n  ✅  ratio=0.33
]

topology_results = {}

for tc in topology_configs:
    print(f'\n{"="*60}')
    print(f'Running: n_agents={tc["n_agents"]}, k={tc["k"]}  ({tc["label"]})')
    print(f'{"="*60}')

    topo_cfg = AblationConfig(
        alpha = cfg.alpha,
        beta  = cfg.beta,
        gamma = cfg.gamma,
        k     = tc['k'],

        n_agents    = tc['n_agents'],
        epochs      = cfg.epochs,
        batch_size  = cfg.batch_size,
        subset_size = cfg.subset_size,
        device      = DEVICE,
        lr          = cfg.lr,
        weight_decay= cfg.weight_decay,

        conditions       = ['sep_coh', 'full_swarm'],
        landscape_at_end = False,
        cka_interval     = 5,
        wandb_mode       = cfg.wandb_mode,
        wandb_project    = cfg.wandb_project,
        checkpoint_dir   = CHECKPOINT_DIR / 'topology_sweep' / tc['label'],
    )

    res = run_ablation(topo_cfg)
    topology_results[tc['label']] = {
        'n': tc['n_agents'],
        'k': tc['k'],
        'results': res,
    }

print('\n✓ Topology sweep complete.')

In [ ]:
# ── Topology sweep results table ─────────────────────────────────────────

print(f'\n{"Config":<10} {"Condition":<12} {"n":>4} {"k":>4} {"k/n":>6} '
      f'{"k÷n?":>6} {"Ens Acc":>8} {"Diversity":>10} {"GAP CKA":>8}')
print('─' * 72)

for label, entry in topology_results.items():
    n, k = entry['n'], entry['k']
    divides = '✅' if n % k == 0 else '❌'
    ratio   = k / n

    for cond, res in entry['results'].items():
        tm      = res['test_metrics']
        gap_cka = '—'
        if res['cka_history']:
            last = res['cka_history'][-1]
            if 'gap' in last:
                gap_cka = f'{last["gap"]["mean_sim"]:.3f}'

        print(f'{label:<10} {cond:<12} {n:>4} {k:>4} {ratio:>6.2f} '
              f'{divides:>6} {tm["ensemble_acc"]:>8.3f} '
              f'{tm["diversity"]:>10.2f} {gap_cka:>8}')
    print()

In [ ]:
# ── Training curves: spot instability visually ────────────────────────────

import matplotlib.pyplot as plt

conditions  = ['sep_coh', 'full_swarm']
n_configs   = len(topology_results)

fig, axes = plt.subplots(len(conditions), n_configs,
                          figsize=(5 * n_configs, 4 * len(conditions)),
                          sharey='row')

for col, (label, entry) in enumerate(topology_results.items()):
    n, k = entry['n'], entry['k']
    divides = '✅' if n % k == 0 else '❌'

    for row, cond in enumerate(conditions):
        ax  = axes[row, col]
        res = entry['results'].get(cond)

        if res and res['history']:
            epochs_  = [d['epoch']     for d in res['history']]
            means    = [d['mean_loss'] for d in res['history']]

            for i in range(len(res['history'][0]['agent_losses'])):
                agent_curve = [d['agent_losses'][i] for d in res['history']]
                ax.plot(epochs_, agent_curve, linewidth=0.8, alpha=0.4)

            ax.plot(epochs_, means, linewidth=2, color='black', label='mean')

        if row == 0:
            ax.set_title(f'{label}\nn={n}, k={k}, k÷n={divides}', fontsize=10)
        if col == 0:
            ax.set_ylabel(f'{cond}\nLoss', fontsize=9)
        ax.set_xlabel('Epoch', fontsize=8)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0.5, 2.1)

fig.suptitle('Topology Sweep — Training Stability (sep_coh & full_swarm)', fontsize=13)
fig.tight_layout()
plt.savefig(str(CHECKPOINT_DIR / 'topology_sweep' / 'training_curves.png'),
            dpi=150, bbox_inches='tight')
plt.show()